# Robotics + RL/DL + Explainability Quick Start

This notebook provides a quick introduction to running MuJoCo experiments with explainability analysis.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import wandb

from stable_baselines3 import PPO
from src.explainability import PolicyAnalyzer

## 1. Create MuJoCo Environment

In [ ]:
# Create environment
env = gym.make('Ant-v4', render_mode='rgb_array')
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

In [ ]:
# Visualize environment
obs, info = env.reset()
frame = env.render()
plt.imshow(frame)
plt.axis('off')
plt.title('Ant Environment')
plt.show()

## 2. Quick Training (Demo)

In [ ]:
# Initialize W&B (optional)
wandb.init(project='robotics-xai-research', name='quickstart-demo', mode='disabled')  # Set mode='online' to enable

In [ ]:
# Train a quick demo model
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=10000)  # Short demo, increase for real training

## 3. Explainability Analysis

In [ ]:
# Collect some observations
observations = []
obs, _ = env.reset()
for _ in range(100):
    observations.append(obs)
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        obs, _ = env.reset()

observations = np.array(observations)

In [ ]:
# Analyze policy
analyzer = PolicyAnalyzer(model, method='saliency')
importance = analyzer.compute_feature_importance(observations)

# Plot feature importance
plt.figure(figsize=(14, 5))
plt.bar(range(len(importance['mean_importance'])), importance['mean_importance'])
plt.xlabel('Observation Dimension')
plt.ylabel('Mean Saliency')
plt.title('Feature Importance in Policy Decisions')
plt.show()

In [ ]:
# Analyze action distribution for a specific state
action_analysis = analyzer.analyze_action_distribution(observations[0])
print("Action Distribution Analysis:")
print(f"  Mean: {action_analysis['action_mean']}")
print(f"  Std: {action_analysis['action_std']}")
print(f"  Entropy: {action_analysis['entropy']}")

## 4. Cleanup

In [ ]:
env.close()
wandb.finish()